# Patterns of Crime in Melbourne, Victoria

## KGG375 Geospatial Analysis

This project investigates the spatial distribution of crime across Melbourne suburbs and explores relationships between crime patterns and population characteristics using geospatial analysis techniques.

Authors: Shaun Knight, Kalvin Pitt
Date: October 2026

## Study Area

The study area consists of suburbs within the Melbourne metropolitan region in Victoria, Australia. Suburb boundary data obtained from Vicmap is used as the spatial framework for analysis.

The aim of this project is to examine the geographical distribution of crime and identify whether spatial clustering exists between neighbouring suburbs.

## Data Acquisition and Preparation

Three datasets were used in this project:

1. Vicmap suburb boundary polygons.
2. Victorian crime statistics.
3. Suburb population data.

These datasets were imported into Python and prepared for subsequent spatial analysis.

## Suburb Boundary Dataset

The suburb boundary dataset was loaded as a GeoDataFrame using GeoPandas. Each polygon represents a Victorian suburb and contains identifying attributes including suburb names and administrative information.

In [44]:
import geopandas as gpd
import pandas as pd

suburbs = gpd.read_file(
    r"Data/VICMapSuburbPolys/mga2020_55/esrishape/user_polygon/user_polygon-0/VMADMIN/AD_LOCALITY_AREA_POLYGON.shp"
)

print(suburbs.head())
print(suburbs.columns)

        UFI      PFI FTYPE_CODE             NAME  NFEAT_ID        NUNIQ_LOC  \
0  69885514  4160577   locality         WANTIRNA       815         WANTIRNA   
1  74680597  4160842   locality           ELTHAM       311           ELTHAM   
2  70741707  4160890   locality  SCOTCHMANS LEAD     16105  SCOTCHMANS LEAD   
3  76275819  4160639   locality         RICHMOND       664         RICHMOND   
4  76275821  4160618   locality    HAWTHORN EAST       392    HAWTHORN EAST   

  STATE  FQID  TASK_ID CRDATE_PFI  SUPER_PFI CRDATE_UFI  LABEL_USE_  \
0   VIC  4712        4 2002-01-14          0 2023-11-30           5   
1   VIC  4712     5043 2002-01-14          0 2025-10-30           5   
2   VIC  4897        4 2002-01-14          0 2024-06-05           5   
3   VIC  4712     5102 2002-01-14          0 2026-04-27           5   
4   VIC  4712     5102 2002-01-14          0 2026-04-27           5   

                                            geometry  
0  POLYGON ((344167.972 5810514.524, 344176

## Population Dataset

Population data was obtained for Victorian suburbs and used to standardise crime counts into population-adjusted crime rates. Standardisation allows meaningful comparison between suburbs of different sizes.

In [45]:
pop = pd.read_csv("Data/Suburb_pop.csv")
crime = pd.read_csv("Data/Crime_rates.csv")

print(pop.columns)
print(crime.columns)

Index(['SAL_CODE', 'SUBURB_NAME', 'TOT_POP', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')
Index(['Year', 'Year ending', 'Local Government Area', 'Postcode',
       'Suburb/Town Name', 'Offence Division', 'Offence Subdivision',
       'Offence Subgroup', 'Offence Count'],
      dtype='str')


## Crime Dataset

Crime data were obtained from the Victorian Crime Statistics Agency. The selected table contains offence counts recorded at the suburb or town level and includes offence classifications, local government areas, and crime counts.

Suburb-level data were selected because they provide a finer spatial resolution than local government area summaries and are therefore more suitable for spatial pattern analysis.

In [46]:
crime = pd.read_csv("Data/Crime_rates.csv",)

crime.head()

,Year,Year ending,Local Government Area,Postcode,Suburb/Town Name,Offence Division,Offence Subdivision,Offence Subgroup,Offence Count
0,2026,March,Alpine,3691,Coral Bank,A Crimes against the person,A20 Assault and related offences,A211 FV Serious assault,1
1,2026,March,Alpine,3691,Coral Bank,B Property and deception offences,B20 Property damage,B21 Criminal damage,1
2,2026,March,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1
3,2026,March,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B49 Other theft,1
4,2026,March,Alpine,3691,Dederang,C Drug offences,C30 Drug use and possession,C32 Drug possession,1


## Data Cleaning and Harmonisation

The crime, population, and boundary datasets were sourced from different organisations and therefore used different geographic definitions and naming conventions.

Prior to analysis, suburb names were standardised to ensure consistency between datasets. Records that could not be matched across all datasets were excluded from further analysis. Only suburbs present in the crime, population, and boundary datasets were retained for subsequent spatial analysis.

In [47]:
# Keep only required columns
pop_summary = pop[
    ["SUBURB_NAME", "TOT_POP"]
].copy()

# Convert population to numeric
pop_summary["TOT_POP"] = pd.to_numeric(
    pop_summary["TOT_POP"],
    errors="coerce"
)

# Remove suburbs with zero population
pop_summary = pop_summary[
    pop_summary["TOT_POP"] > 0
]

In [48]:
suburbs["NAME"] = (
    suburbs["NAME"]
    .str.upper()
    .str.strip()
)

pop["SUBURB NAME"] = (
    pop["SUBURB_NAME"]
    .str.upper()
    .str.replace(" (VIC.)", "", regex=False)
    .str.strip()
)

crime["Suburb/Town Name"] = (
    crime["Suburb/Town Name"]
    .str.upper()
    .str.strip()
)

crime_suburbs = set(crime["Suburb/Town Name"].dropna())
pop_suburbs = set(pop["SUBURB NAME"].dropna())
poly_suburbs = set(suburbs["NAME"].dropna())

print("Crime suburbs:", len(crime_suburbs))
print("Population suburbs:", len(pop_suburbs))
print("Polygon suburbs:", len(poly_suburbs))

all_three = crime_suburbs & pop_suburbs & poly_suburbs

print("Matching suburbs:", len(all_three))

Crime suburbs: 2867
Population suburbs: 2944
Polygon suburbs: 1274
Matching suburbs: 1251


## Matching Geographic Units

After standardising suburb names, the overlap between datasets was assessed.

A total of 1,251 suburbs were present in all three datasets and were retained for analysis. Because spatial analysis requires both attribute and boundary information, the common set of suburbs was used as the final study dataset.

Suburbs without matching records across all datasets were excluded from subsequent analysis.

In [49]:
crime = crime[
    crime["Suburb/Town Name"].isin(all_three)
]

pop = pop[
    pop["SUBURB NAME"].isin(all_three)
]

suburbs = suburbs[
    suburbs["NAME"].isin(all_three)
]

print(len(crime))
print(len(pop))
print(len(suburbs))

266445
1251
1251


## Aggregation of Crime Data

The crime dataset contained individual offence categories and subcategories for each suburb. To enable suburb-level spatial analysis, offence counts were aggregated to produce a single total crime count for each suburb.

This aggregation ensured that each suburb was represented by one crime value that could be directly compared with population and spatial boundary datasets.

In [50]:
crime["Offence Count"] = pd.to_numeric(
    crime["Offence Count"],
    errors="coerce"
)

crime_summary = (
    crime
    .groupby("Suburb/Town Name", as_index=False)
    ["Offence Count"]
    .sum()
)

crime_summary.columns = [
    "SUBURB",
    "TOTAL_CRIME"
]

crime_summary.head()

,SUBURB,TOTAL_CRIME
0,ABBOTSFORD,14723.0
1,ABERFELDIE,1410.0
2,ACHERON,43.0
3,ADA,6.0
4,ADAMS ESTATE,57.0


In [51]:
pop_summary = pop[
    ["SUBURB NAME", "TOT_POP"]
].copy()

pop_summary.columns = [
    "SUBURB",
    "POPULATION"
]

pop_summary["POPULATION"] = pd.to_numeric(
    pop_summary["POPULATION"],
    errors="coerce"
)

# Remove zero and missing populations
pop_summary = pop_summary[
    pop_summary["POPULATION"] > 50
]

print((pop_summary["POPULATION"] < 50).sum())

0


## Merging Crime and Population Data

The aggregated crime dataset was joined with suburb population data using standardised suburb names. This enabled the calculation of population-adjusted crime rates, allowing meaningful comparisons between suburbs of different sizes.

In [52]:
crime_pop = crime_summary.merge(
    pop_summary,
    on="SUBURB",
    how="inner"
)

crime_pop.head()

,SUBURB,TOTAL_CRIME,POPULATION
0,ABBOTSFORD,14723.0,9088.0
1,ABERFELDIE,1410.0,3925.0
2,ACHERON,43.0,146.0
3,ADAMS ESTATE,57.0,83.0
4,ADDINGTON,13.0,65.0


In [53]:
print(len(crime_summary))
print(len(pop_summary))
print(len(crime_pop))

1251
1094
1094


In [54]:
crime_pop["CRIME_RATE_1000"] = (
    crime_pop["TOTAL_CRIME"]
    / crime_pop["POPULATION"]
) * 1000

crime_pop.head()

,SUBURB,TOTAL_CRIME,POPULATION,CRIME_RATE_1000
0,ABBOTSFORD,14723.0,9088.0,1620.048415
1,ABERFELDIE,1410.0,3925.0,359.235669
2,ACHERON,43.0,146.0,294.520548
3,ADAMS ESTATE,57.0,83.0,686.746988
4,ADDINGTON,13.0,65.0,200.000000


## Spatial Data Integration

The crime and population datasets were joined to the suburb boundary dataset using standardised suburb names. The resulting GeoDataFrame contains geometry, total crime counts, population, and crime rates for each suburb and forms the basis of the spatial analysis.

In [55]:
crime_gdf = suburbs.merge(
    crime_pop,
    left_on="NAME",
    right_on="SUBURB",
    how="inner"
)

print(len(crime_gdf))
crime_gdf.head()

1094


,UFI,PFI,FTYPE_CODE,NAME,NFEAT_ID,NUNIQ_LOC,STATE,FQID,TASK_ID,CRDATE_PFI,SUPER_PFI,CRDATE_UFI,LABEL_USE_,geometry,SUBURB,TOTAL_CRIME,POPULATION,CRIME_RATE_1000
0,69885514,4160577,locality,WANTIRNA,815,WANTIRNA,VIC,4712,4,2002-01-14,0,2023-11-30,5,"POLYGON ((344167.972 5810514.524, 344176.151 5...",WANTIRNA,7570.0,14237.0,531.713142
1,74680597,4160842,locality,ELTHAM,311,ELTHAM,VIC,4712,5043,2002-01-14,0,2025-10-30,5,"POLYGON ((337362.264 5827158.936, 337364.301 5...",ELTHAM,8697.0,18847.0,461.452751
2,70741707,4160890,locality,SCOTCHMANS LEAD,16105,SCOTCHMANS LEAD,VIC,4897,4,2002-01-14,0,2024-06-05,5,"POLYGON ((221062.531 5827986.079, 221084.35 58...",SCOTCHMANS LEAD,5.0,105.0,47.619048
3,76275819,4160639,locality,RICHMOND,664,RICHMOND,VIC,4712,5102,2002-01-14,0,2026-04-27,5,"POLYGON ((323182.744 5813396.322, 323237.771 5...",RICHMOND,42383.0,28587.0,1482.596985
4,76275821,4160618,locality,HAWTHORN EAST,392,HAWTHORN EAST,VIC,4712,5102,2002-01-14,0,2026-04-27,5,"POLYGON ((329187.275 5812674.926, 329183.296 5...",HAWTHORN EAST,7940.0,14834.0,535.256842


## Treatment of Small Population Areas

Initial analysis identified a number of suburbs with very small resident populations. Crime rates calculated for these locations were highly unstable because small population denominators can produce disproportionately large per-capita crime rates.

To reduce the influence of statistical outliers, suburbs with populations of 50 residents or fewer were excluded from further analysis. This reduced the dataset from 1,251 to 1,094 suburbs while retaining the majority of the study area.

## Population and Crime Relationship

A Pearson correlation analysis was used to examine the relationship between suburb population and crime rate. This provides a measure of the strength and direction of association between population size and recorded crime rates.

In [56]:
from scipy.stats import pearsonr

corr, p = pearsonr(
    crime_gdf["POPULATION"],
    crime_gdf["CRIME_RATE_1000"]
)

print("Correlation coefficient:", corr)
print("P-value:", p)

Correlation coefficient: -0.019254747095139167
P-value: 0.524649596735456


In [57]:
q95 = crime_gdf["CRIME_RATE_1000"].quantile(0.95)

hotspots = crime_gdf[
    crime_gdf["CRIME_RATE_1000"] >= q95
]

hotspots[
    ["SUBURB", "CRIME_RATE_1000"]
].sort_values(
    "CRIME_RATE_1000",
    ascending=False
).head(20)

,SUBURB,CRIME_RATE_1000
558,MELBOURNE AIRPORT,114359.375000
589,DANDENONG SOUTH,96496.000000
600,LAVERTON NORTH,62302.521008
598,SOUTH WHARF,44070.422535
1018,GRANGEFIELDS,17553.030303
717,BAKERY HILL,7577.777778
788,KEILOR NORTH,6955.223881
199,GEELONG,5787.644123
222,BALLARAT CENTRAL,4838.973596
887,BONNIE BROOK,4567.567568


## Hotspot Analysis

Crime hotspots were identified using the 95th percentile of the crime-rate distribution. Suburbs with crime rates greater than or equal to the 95th percentile were classified as hotspots.

This percentile-based approach highlights areas with exceptionally high crime rates relative to the broader study area and supports the identification of spatial concentrations of crime.

In [58]:
print("95th percentile:", q95)

print("Number of hotspots:", len(hotspots))

print(
    "Percentage of suburbs:",
    round(
        len(hotspots) / len(crime_gdf) * 100,
        2
    ),
    "%"
)

95th percentile: 1744.8184428783236
Number of hotspots: 55
Percentage of suburbs: 5.03 %


In [59]:
crime_gdf["HOTSPOT"] = (
    crime_gdf["CRIME_RATE_1000"] >= q95
)

crime_gdf.groupby("HOTSPOT")[
    ["POPULATION", "TOTAL_CRIME", "CRIME_RATE_1000"]
].describe()

POPULATION                                                          \
             count         mean           std   min    25%     50%     75%   
HOTSPOT                                                                      
False       1039.0  5027.815207   8414.855463  51.0  188.5   979.0  6530.5   
True          55.0  6295.054545  10184.604408  64.0  242.5  1979.0  8317.5   

                 TOTAL_CRIME                ...                     \
             max       count          mean  ...      75%       max   
HOTSPOT                                     ...                      
False    66781.0      1039.0   3345.250241  ...   3635.5   53016.0   
True     54941.0        55.0  15732.945455  ...  24227.0  141549.0   

        CRIME_RATE_1000                                                       \
                  count         mean           std          min          25%   
HOTSPOT                                                                        
False            1039.0   540.546182    319.912056    47.619048   309.346457   
True               55.0  8551.270024  21492.953595  1746.761387  2008.133474   

                                                  
                 50%          75%            max  
HOTSPOT                                           
False     461.073318   695.868717    1743.772242  
True     2348.369565  3278.469453  114359.375000  

[2 rows x 24 columns]

In [60]:
results_summary = pd.DataFrame({
    "Metric": [
        "Total suburbs",
        "Hotspot suburbs",
        "95th percentile threshold",
        "Population-crime correlation",
        "P-value"
    ],
    "Value": [
        len(crime_gdf),
        len(hotspots),
        round(q95, 2),
        round(corr, 3),
        round(p, 3)
    ]
})

results_summary

,Metric,Value
0,Total suburbs,1094.000
1,Hotspot suburbs,55.000
2,95th percentile threshold,1744.820
3,Population-crime correlation,-0.019
4,P-value,0.525


In [61]:
hotspots[
    ["SUBURB", "POPULATION", "TOTAL_CRIME", "CRIME_RATE_1000"]
].sort_values(
    "CRIME_RATE_1000",
    ascending=False
)

,SUBURB,POPULATION,TOTAL_CRIME,CRIME_RATE_1000
558,MELBOURNE AIRPORT,64.0,7319.0,114359.375000
589,DANDENONG SOUTH,125.0,12062.0,96496.000000
600,LAVERTON NORTH,119.0,7414.0,62302.521008
598,SOUTH WHARF,71.0,3129.0,44070.422535
1018,GRANGEFIELDS,132.0,2317.0,17553.030303
717,BAKERY HILL,180.0,1364.0,7577.777778
788,KEILOR NORTH,67.0,466.0,6955.223881
199,GEELONG,5811.0,33632.0,5787.644123
222,BALLARAT CENTRAL,5378.0,26024.0,4838.973596
887,BONNIE BROOK,333.0,1521.0,4567.567568


## Interpretation of Hotspots

Several hotspot suburbs were associated with major industrial, commercial, and transport precincts rather than traditional residential suburbs. Examples included Melbourne Airport, Laverton North, Dandenong South, and South Wharf.

These areas experience large daily movements of workers, visitors, freight vehicles, and commercial activity. Consequently, resident population may not accurately represent the population exposed to crime opportunities within these locations.

This finding is consistent with the statistical analysis, which found no significant relationship between residential population and crime rate.